# 2D Ising MCMC (Glauber Dynamics)

This notebook provides a ipython notebook template for the computer experiment described in `lec5.pdf`:
- lattice size `L=32`, coupling `J=1`
- run `5000` sweeps at temperatures `T in {1.5, 2.3, 3.5}`
- compare magnetization traces and autocorrelation

Notation reminder: `beta = 1 / T`.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation


In [ ]:
# Use Python type support
SpinArray = npt.NDArray[np.int_]


def init_spins(m: int, n: int, rng: np.random.Generator) -> SpinArray:
    """Initialize an (m, n) Ising lattice with spins in {-1, +1}."""
    return np.where(rng.random((m, n)) < 0.5, -1, 1).astype(np.int_)


def sweep_glauber(spins: SpinArray, J: float, h: float, beta: float, rng: np.random.Generator) -> SpinArray:
    """Run one Glauber sweep (m*n random-site updates) with periodic boundaries."""
    m, n = spins.shape
    for _ in range(m * n):
        i = int(rng.integers(0, m))
        j = int(rng.integers(0, n))
        # Add code here to compute effective field for spin[i,j] and resample it
    return spins


def magnetization(spins: SpinArray) -> float:
    """Return magnetization per spin M = mean(sigma)."""
    return float(np.mean(spins))


def energy_per_spin(spins: SpinArray, J: float, h: float) -> float:
    """Return energy per spin U = E / N with periodic boundaries."""
    m, n = spins.shape
    interaction = 0.0
    for i in range(m):
        interaction += np.dot(spins[i,:],spins[(i + 1) % m,:])
    for j in range(n):
        interaction += np.dot(spins[:,j],spins[:,(j + 1) % m])
    field = np.sum(spins)
    E = -J * interaction - h * field
    return float(E / spins.size)


def run_chain(
    L: int,
    sweeps: int,
    temperature: float,
    J: float = 1.0,
    h: float = 0.0,
    seed: int = 0,
    keep_frames: bool = False,
) -> dict[str, object]:
    """Simulate 2D Ising Glauber dynamics and record observables by sweep."""
    beta = 1.0 / temperature
    rng = np.random.default_rng(seed)

    spins = init_spins(L, L, rng)
    frames: list[SpinArray] = [spins.copy()] if keep_frames else []
    m_trace: list[float] = []
    u_trace: list[float] = []

    for _ in range(sweeps):
        sweep_glauber(spins, J=J, h=h, beta=beta, rng=rng)
        m_trace.append(magnetization(spins))
        u_trace.append(energy_per_spin(spins, J=J, h=h))
        if keep_frames:
            frames.append(spins.copy())

    return {
        "spins": spins,
        "M": np.asarray(m_trace, dtype=float),
        "U": np.asarray(u_trace, dtype=float),
        "frames": frames,
        "beta": beta,
        "temperature": temperature,
        "J": J,
        "h": h,
        "seed": seed,
    }


In [ ]:
def animate_spins(frames: list[SpinArray], title: str = "Spin configuration", interval_ms: int = 50) -> FuncAnimation:
    """Create a matplotlib animation for a sequence of Ising lattices."""
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(frames[0], cmap="gray", vmin=-1, vmax=1, interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    title_text = ax.set_title(f"{title} (sweep 0)")

    def update(k: int):
        im.set_data(frames[k])
        title_text.set_text(f"{title} (sweep {k})")
        return im, title_text

    anim = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval_ms,
        blit=False,
        repeat=False,
    )
    plt.close(fig)
    return anim


## Demo run parameters

- random initialization
- `L=32`, `J=1`, `h=0`
- `S=800` sweeps at `T=2.3` (`beta=1/2.3`)
- animate spin configuration after each sweep


In [ ]:
demo = run_chain(L=32, sweeps=800, temperature=2.3, J=1.0, h=0.0, seed=1, keep_frames=True)
anim = animate_spins(demo["frames"], title=f"2D Ising (L=32, beta={demo['beta']:.3f}, J=1, h=0)", interval_ms=40)
display(HTML(anim.to_jshtml()))


In [ ]:
plt.figure(figsize=(4.5, 4.5))
plt.imshow(demo["spins"], cmap="gray", vmin=-1, vmax=1, interpolation="nearest")
plt.title("Final spin configuration")
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()


## Illustrative Experiment

Run `5000` sweeps for `T in {1.5, 2.3, 3.5}` and compare:
- magnetization traces `M_t`
- autocorrelation of magnetization


In [ ]:
L = 32
J = 1.0
h = 0.0
sweeps = 5000
temperatures = [1.5, 2.3, 3.5]
base_seed = 123

results: dict[float, dict[str, object]] = {}
for k, T in enumerate(temperatures):
    results[T] = run_chain(L=L, sweeps=sweeps, temperature=T, J=J, h=h, seed=base_seed + k, keep_frames=False)

print("Completed runs for temperatures:", temperatures)


In [ ]:
fig, axes = plt.subplots(len(temperatures), 1, figsize=(10, 7), sharex=True)
for ax, T in zip(axes, temperatures):
    M = results[T]["M"]
    ax.plot(M, lw=1.0)
    ax.axhline(0.0, color="black", lw=0.8, ls="--", alpha=0.6)
    ax.set_ylabel(f"M (T={T})")
axes[-1].set_xlabel("Sweep")
fig.suptitle("Magnetization trace by temperature")
fig.tight_layout()
plt.show()


In [ ]:
def autocorrelation(x: npt.ArrayLike, max_lag: int) -> np.ndarray:
    """Compute the normalized autocorrelation of a 1D numpy array."""
    x_arr = np.asarray(x, dtype=float)
    x_centered = x_arr - np.mean(x_arr)
    acf = np.correlate(x_centered, x_centered, mode='full')
    acf = acf[len(acf)//2:]
    if acf[0] > 0:
        acf /= acf[0]
    return acf[:max_lag]


max_lag = 200
fig, ax = plt.subplots(figsize=(8, 4.5))
for T in temperatures:
    acf = autocorrelation(results[T]["M"], max_lag=max_lag)
    ax.plot(acf, label=f"T={T}")
ax.set_xlabel("Lag (sweeps)")
ax.set_ylabel("Autocorrelation of M")
ax.set_title("Magnetization autocorrelation")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
burn_in = 1000
print(f"Burn-in used for summary: {burn_in} sweeps\n")
for T in temperatures:
    M = results[T]["M"][burn_in:]
    U = results[T]["U"][burn_in:]
    print(
        f"T={T:>3}: beta={1.0/T:.3f}, mean(M)={np.mean(M): .4f}, mean(|M|)={np.mean(np.abs(M)): .4f}, mean(U)={np.mean(U): .4f}"
    )
